# step B — directive-token (지침 속 'camelCase' 토큰을 보나?)

**대응 RQ:** RQ2 관측 보강. stepB pool-500은 지침을 **문장 통째(37토큰)** 로만 쟀다 → '지침 어텐션 평탄'. 하지만 그 안의 **표기 지시어 토큰**('camelCase')을 따로 본 적이 없다.

**무엇을 하나** — 지침 안의 **표기 단어**(`instr_target_word`='camelCase' 등)를 별도 span으로 잡아, 그 **토큰당 어텐션**을 코드 이름 토큰·지침 전체와 비교한다.

**왜** — Spotlight(지침 어텐션 증폭)가 유효하려면 '지침이 덜 보여야' 한다. 지시어 토큰이 잘 보이면 Spotlight는 부족하지 않은 걸 키우는 헛다리(우리 주장 강화); 훨씬 낮으면 문장은 읽되 핵심 단어를 스킵 → Spotlight 여지(→ step4 인과).

**하네스 변경:** `observe_generation_query`에 `notation_spans` 추가 — 표기 단어를 span으로 관측(코드/과제와 겹치지 않는 고유 토큰). 측정 로직은 동일.

설계: `docs/stepB/scaleup-500.md` §, qna Q6(예정). 결과는 `results/stepB_directive/`(기존 294와 분리).

> **규모:** 기본 7 spec × 10 block = 70 조건(지시어 어텐션은 조건마다 나오니 적은 블록으로 층 프로파일 충분). 더 보려면 `N_BLOCKS_USE` ↑. eager 필수. 재개 가능.

In [ ]:
# 환경 설정
!pip install -q transformers accelerate torch matplotlib pandas numpy
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')
SEED=0; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepB/directive-token
!git checkout stepB/directive-token
!git pull --quiet origin stepB/directive-token
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — stepB와 동일(View1 flip + View2 스윕), 블록 축소(지시어 관측용)
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)
from harness.tasks import NAME_PAIR_POOL

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
REF_FRAC = 0.7
N_FUNCTIONS = 12
N_BLOCKS_USE = 10                               # 지시어 어텐션은 조건마다 나옴 → 적어도 충분(원하면 ↑, 최대 42)
BLOCKS = list(range(N_BLOCKS_USE))
SEEDS = [0]

FLIP  = [(Notation.CAMEL, 6), (Notation.SNAKE, 6)]
SWEEP = [(Notation.CAMEL, n) for n in (4, 3, 2, 1, 0)]
SPECS = list(dict.fromkeys(FLIP + SWEEP))

def make(target, n, b, s):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=n, n_functions=N_FUNCTIONS, composition=Composition.POOL, pool_block=b),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target), seed=s)

conditions = [make(t, n, b, s) for (t, n) in SPECS for b in BLOCKS for s in SEEDS]
PREDICTION = ('지시어 토큰(camelCase)의 토큰당 어텐션이 코드 토큰과 비슷/이상이면 지침 부족 아님(Spotlight 헛다리); '
              '훨씬 낮으면 핵심 단어 스킵(Spotlight 여지).')
print(f'{len(conditions)} 조건 = {len(SPECS)} spec x {len(BLOCKS)} block x {len(SEEDS)} seed')

In [ ]:
# 실행 — observe(지시어 span 자동 포함). 즉시 저장(재개).
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

STEP = 'stepB_directive'
handle = load_model(MODEL, attn_implementation='eager')   # 어텐션 가중치 필수
REF_LAYER = int(handle.num_layers * REF_FRAC)
print('layers:', handle.num_layers, '| 기준층 L%d' % REF_LAYER)

new = skipped = 0
for i, c in enumerate(conditions, 1):
    p = result_path(c, step=STEP)
    if p.exists(): skipped += 1
    else:
        out = run(c, handle=handle, mode='observe')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ2', prediction=PREDICTION))
        new += 1
    if i % 10 == 0 or i == len(conditions):
        print(f'[{i}/{len(conditions)}] 새 {new} / 건너뜀 {skipped}')
print('완료.')

In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
records = [load_result(result_path(c, step=STEP)) for c in conditions]
print('로드:', len(records), '-> results/'+STEP+'/')

In [ ]:
# 요약 — 지시어 토큰 vs 코드 토큰 vs 지침전체: 토큰당 어텐션 (핵심 질문)
import pandas as pd, numpy as np, matplotlib.pyplot as plt

# View1(균형 6/6, 지침=camel)에서 층별 토큰당 어텐션
SPANS = {'instr_target_word':'directive word (camelCase)', 'instr_viol_word':'other word (snake_case)',
         'instruction':'instruction (whole)', 'code_camel':'code camel tok', 'code_snake':'code snake tok'}
c6 = [r for r in records if r.condition.instruction.target_notation.value=='camel' and r.condition.preceding.n_compliant==6]
NL = max(int(L) for r in records for L in r.metrics.per_layer)+1
REF_LAYER = int(NL*0.7)

def pertok(recs, span, L):
    vals=[]
    for r in recs:
        pl=r.metrics.per_layer.get(L, r.metrics.per_layer.get(int(L),{}))
        a=pl.get(f'{span}__attention_weight'); n=r.metrics.extra['span_token_counts'].get(span)
        if a is not None and n: vals.append(a/n)
    return float(np.mean(vals)) if vals else np.nan

print(f'=== 토큰당 어텐션 @L{REF_LAYER} (지침=camel, 균형 6/6) ===')
rows=[]
for sp,lab in SPANS.items():
    ntok=int(np.mean([r.metrics.extra['span_token_counts'].get(sp,0) for r in c6]))
    rows.append({'span':lab, 'per_token_attn':round(pertok(c6,sp,REF_LAYER),5), 'n_tokens':ntok})
tab=pd.DataFrame(rows); print(tab.to_string(index=False))
d=tab.set_index('span')['per_token_attn']
print(f"\n지시어(camelCase) 토큰당 {d['directive word (camelCase)']:.5f} vs 코드 camel 토큰 {d['code camel tok']:.5f}")
print('=> 지시어가 코드 토큰만큼/이상: 지침 부족 아님(Spotlight 헛다리) / 훨씬 낮음: 핵심단어 스킵(Spotlight 여지)')

# 층별 궤적
layers=list(range(NL))
plt.rcParams.update({'font.size':10,'axes.grid':True,'grid.alpha':.3})
fig,ax=plt.subplots(figsize=(8,4))
for sp,lab,c in [('instr_target_word','directive word (camelCase)','#B0392B'),
                 ('code_camel','code camel tok','#2E7D52'),
                 ('code_snake','code snake tok','#C6771A'),
                 ('instruction','instruction (whole)','#7B3FA0')]:
    ax.plot(layers,[pertok(c6,sp,L) for L in layers],marker='.',ms=4,label=lab,color=c)
ax.axvline(REF_LAYER,color='#888',ls=':',lw=1)
ax.set_xlabel('layer'); ax.set_ylabel('attention PER token'); ax.set_title('Directive word vs code tokens (per-token)')
ax.legend(fontsize=8,loc='upper center',bbox_to_anchor=(0.5,-0.18),ncol=2,frameon=False)
plt.tight_layout(rect=[0,0.08,1,1]); plt.savefig('stepB_directive_summary.png',dpi=120); plt.show()

In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive('stepB_directive_results', 'zip', 'results/'+STEP)
try:
    from google.colab import files; files.download('stepB_directive_results.zip')
except Exception as e:
    print('Colab 아님(수동): stepB_directive_results.zip', e)